<a href="https://colab.research.google.com/github/costpetrides/Box_Model/blob/main/Plots.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Box Model — all CSV scenarios

Automatically discover **every CSV in the repository**, produce one dual-axis plot per scenario (O₃, NO, NO₂, CO; OH on the right), and plot **each selected species across all scenarios** for comparison.

This notebook **only reads GitHub**. All plots and the ZIP are saved in the current Jupyter/Colab session; nothing is committed or uploaded to the repository.

In [ ]:
from pathlib import Path
from io import BytesIO
from zipfile import ZipFile, ZIP_DEFLATED
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from matplotlib.ticker import MultipleLocator, FuncFormatter
from IPython.display import display

# Configuration
REPO = "costpetrides/Box_Model"
BRANCH = "main"
OUTPUT_DIR = Path("box_model_plots")
DPI = 300
HOUR_INTERVAL = 2

# Change these lists to choose which chemicals appear in each plot.
PRIMARY_SPECIES = {
    "O3":  (r"O$_3$", "tab:blue", "o"),
    "NO":  ("NO", "tab:orange", "s"),
    "NO2": (r"NO$_2$", "tab:green", "^"),
    "CO":  ("CO", "tab:purple", "D"),
}
SECONDARY_SPECIES = "OH"
COMPARISON_SPECIES = [*PRIMARY_SPECIES, SECONDARY_SPECIES]

# Enable/disable figure groups.
PLOT_EACH_SCENARIO = True
PLOT_SCENARIO_COMPARISONS = True
SAVE_PNG = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Discover and load the CSVs

The GitHub directory is read **once**. CSVs are then downloaded directly using their URLs (including filenames with spaces and `%`). They use `;` as delimiter and `,` as decimal separator. The `Time` field is in **minutes since midnight**; e.g. `360 → 06:00`.

In [ ]:
session = requests.Session()
session.headers.update({"Accept": "application/vnd.github+json", "User-Agent": "Box-Model-Plots"})

api_url = f"https://api.github.com/repos/{REPO}/contents"
response = session.get(api_url, params={"ref": BRANCH}, timeout=30)
response.raise_for_status()

files = sorted(
    (item for item in response.json()
     if item["type"] == "file" and item["name"].lower().endswith(".csv")),
    key=lambda item: item["name"].casefold()
)
if not files:
    raise ValueError("No CSV files were found in the repository root.")


def read_csv(item):
    result = session.get(item["download_url"], timeout=30)
    result.raise_for_status()
    df = pd.read_csv(BytesIO(result.content), sep=";", decimal=",")
    df.columns = df.columns.str.strip()
    if "Time" not in df.columns:
        raise ValueError("Missing Time column")
    df["Time"] = pd.to_numeric(df["Time"], errors="coerce")
    df = df.dropna(subset=["Time"]).sort_values("Time").reset_index(drop=True)
    if df.empty:
        raise ValueError("No valid time rows")
    return df


datasets, errors = {}, {}
for item in files:
    try:
        datasets[item["name"]] = read_csv(item)
    except (requests.RequestException, ValueError, pd.errors.ParserError) as exc:
        errors[item["name"]] = str(exc)

if not datasets:
    raise RuntimeError(f"Failed to load all CSV files: {errors}")

print(f"Loaded {len(datasets)} of {len(files)} CSV scenarios.")
display(pd.DataFrame([
    {"Scenario": name, "Time points": len(df), "Data columns": len(df.columns) - 1}
    for name, df in datasets.items()
]))
if errors:
    print("Skipped files:", errors)

## 2. Reusable plot functions

The original dual-axis style is preserved. The time axis displays `HH:MM` (including `00:00` at 24 hours), and legends are placed outside the data area.

In [ ]:
def format_time(hours, _position):
    total_minutes = round(hours * 60)
    h, m = divmod(total_minutes, 60)
    return f"{h % 24:02d}:{m:02d}"


def style_time_axis(ax):
    ax.set_xlabel("Time of Day (HH:MM)")
    ax.xaxis.set_major_locator(MultipleLocator(HOUR_INTERVAL))
    ax.xaxis.set_major_formatter(FuncFormatter(format_time))
    ax.grid(alpha=0.25)


def scenario_label(filename):
    return Path(filename).stem.removesuffix("-Table 1")


def safe_name(label):
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", label).strip("_")


def plot_scenario(filename, df):
    fig, ax = plt.subplots(figsize=(12, 6))
    time = df["Time"] / 60
    marker_step = max(1, len(df) // 20)
    lines = []

    for column, (label, color, marker) in PRIMARY_SPECIES.items():
        if column not in df.columns:
            continue
        values = pd.to_numeric(df[column], errors="coerce")
        line, = ax.plot(time, values, label=label, color=color, lw=2,
                        marker=marker, ms=4, markevery=marker_step)
        lines.append(line)

    ax.set_ylabel("Concentration [ppm]")
    style_time_axis(ax)
    ax.set_xlim(time.min(), time.max())
    if lines and all(np.nanmin(line.get_ydata()) >= 0 for line in lines):
        ax.set_ylim(bottom=0)

    if SECONDARY_SPECIES in df.columns:
        ax2 = ax.twinx()
        oh = pd.to_numeric(df[SECONDARY_SPECIES], errors="coerce")
        line, = ax2.plot(time, oh, label=SECONDARY_SPECIES,
                         color="tab:red", lw=2, ls="--", marker="d",
                         ms=4, markevery=marker_step)
        lines.append(line)
        ax2.set_ylabel(f"{SECONDARY_SPECIES} concentration [ppm]", color="tab:red")
        ax2.tick_params(axis="y", labelcolor="tab:red")
        ax2.grid(False)
        if oh.notna().any() and oh.min() >= 0:
            ax2.set_ylim(bottom=0)

    if not lines:
        plt.close(fig)
        print(f"No selected species found in {filename}; skipped plot.")
        return

    ax.set_title(scenario_label(filename), pad=15)
    fig.legend(lines, [line.get_label() for line in lines],
               loc="lower center", bbox_to_anchor=(0.5, 0.01), ncol=5,
               frameon=False)
    fig.subplots_adjust(bottom=0.18)
    if SAVE_PNG:
        fig.savefig(OUTPUT_DIR / f"scenario_{safe_name(scenario_label(filename))}.png",
                    dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def plot_comparison(species):
    fig, ax = plt.subplots(figsize=(12, 6))
    colours = plt.get_cmap("tab20")(np.linspace(0, 1, max(1, len(datasets))))
    plotted = 0

    for colour, (filename, df) in zip(colours, datasets.items()):
        if species not in df.columns:
            continue
        values = pd.to_numeric(df[species], errors="coerce")
        baseline = filename.startswith("TESTout1-")
        ax.plot(df["Time"] / 60, values, label=scenario_label(filename),
                color=colour, lw=2.6 if baseline else 1.7,
                ls="-" if baseline else "--", alpha=0.9)
        plotted += 1

    if not plotted:
        plt.close(fig)
        print(f"{species} not found in the CSVs; skipped comparison.")
        return

    style_time_axis(ax)
    ax.set_title(f"{species}: comparison of all CSV scenarios", pad=15)
    ax.set_ylabel(f"{species} concentration [ppm]")
    if all(df[species].min() >= 0 for df in datasets.values() if species in df):
        ax.set_ylim(bottom=0)
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1),
              fontsize=9, frameon=False)
    fig.tight_layout()
    if SAVE_PNG:
        fig.savefig(OUTPUT_DIR / f"comparison_{safe_name(species)}.png",
                    dpi=DPI, bbox_inches="tight")
    plt.show()
    plt.close(fig)

## 3. Plot every CSV scenario

One plot for each CSV, with **O₃, NO, NO₂, CO** on the left axis and **OH** on the right. This produces 12 plots for the current repository.

In [ ]:
if PLOT_EACH_SCENARIO:
    for name, df in datasets.items():
        plot_scenario(name, df)

## 4. Compare all scenarios by chemical species

One plot per selected species, containing one line for **each CSV scenario**. The `TESTout1` baseline is drawn with a thicker solid line.

In [ ]:
if PLOT_SCENARIO_COMPARISONS:
    for species in COMPARISON_SPECIES:
        plot_comparison(species)

## 5. Export PNGs as one ZIP

When `SAVE_PNG = True`, the figures are saved in `box_model_plots/`, and this cell also creates a ZIP for easy downloading from Colab's file browser. The notebook never writes to GitHub.

In [ ]:
if SAVE_PNG:
    images = sorted(OUTPUT_DIR.glob("*.png"))
    zip_path = Path("box_model_all_plots.zip")
    with ZipFile(zip_path, "w", ZIP_DEFLATED) as archive:
        for image in images:
            archive.write(image, arcname=image.name)
    print(f"Saved {len(images)} PNG figures in {OUTPUT_DIR}/")
    print(f"ZIP file: {zip_path.resolve()}")
else:
    print("SAVE_PNG is False: figures were displayed but not exported.")